In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('/content/drive/MyDrive/llm_price_performance_tracker_2026-03-31.csv')
df =df.dropna(axis=0, how='any', subset = ['blended_cost_usd_per_1m', 'chatbot_arena_elo'])
print(df.head())

               model_name              model_slug provider provider_slug  \
0         GPT-5.4 (xhigh)                 gpt-5-4   OpenAI        openai   
1  Gemini 3.1 Pro Preview  gemini-3-1-pro-preview   Google        google   
5         GPT-5.2 (xhigh)                 gpt-5-2   OpenAI        openai   
6       GLM-5 (Reasoning)                   glm-5     Z AI           zai   
9             MiMo-V2-Pro             mimo-v2-pro   Xiaomi        xiaomi   

                                  aa_id  aa_intelligence_index  \
0  a89c4b28-2d8c-456e-88ea-255fb51fd2b6                   57.2   
1  bbd93ebe-80da-4594-bb19-61e69d0331df                   57.2   
5  498862c3-f9ac-49d2-852f-16a02bb0c38f                   51.3   
6  40663ad2-b218-471e-bdd4-a1e0c2360e2b                   49.8   
9  5d8183dc-24f4-46c5-a1d0-d937de149364                   49.2   

   aa_coding_index  aa_math_index  composite_benchmark  mmlu_pro  ...  \
0             57.3            NaN                63.40       NaN  ...   


In [27]:
top3 = (df.dropna(subset=["chatbot_arena_elo"]).sort_values("chatbot_arena_elo", ascending=False).head(3)["model_name"].tolist())
print(f"\nTop 3 by ELO: {top3}")


Top 3 by ELO: ['Claude Opus 4.6 (Non-reasoning, High Effort)', 'Gemini 3.1 Pro Preview', 'Gemini 3 Pro Preview (high)']


In [28]:
FEATURES = ["aa_intelligence_index", "aa_coding_index", "aa_math_index", "composite_benchmark", "output_tokens_per_second", "time_to_first_token_s"]
TARGET = "blended_cost_usd_per_1m"
numeric_df = df[FEATURES + [TARGET]].dropna()
X = numeric_df[FEATURES]
y = numeric_df[TARGET]

In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [30]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [31]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

In [32]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2  = r2_score(y_test, y_pred)

print(f"\nMSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}  (in $/1M tokens)")
print(f"R²   : {r2:.4f}")


MSE  : 3.9764
RMSE : 1.9941  (in $/1M tokens)
R²   : -0.4806


In [33]:
coef_df = pd.DataFrame({"feature":FEATURES, "coefficient": model.coef_}).sort_values("coefficient", key=abs, ascending=False)

print("\nFeature coefficients (scaled):")
print(coef_df.to_string(index=False))


Feature coefficients (scaled):
                 feature  coefficient
           aa_math_index    -2.766803
     composite_benchmark     2.087784
   aa_intelligence_index     0.990745
output_tokens_per_second    -0.983572
         aa_coding_index     0.212330
   time_to_first_token_s    -0.063422


In [44]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)
top3_df = df[df["model_name"].isin(top3)][FEATURES + ["model_name", TARGET]]
top3_imputed = imputer.transform(top3_df[FEATURES])
top3_scaled  = scaler.transform(top3_imputed)
top3_df = top3_df.copy()
top3_df["predicted_cost"] = model.predict(top3_scaled)
print("\nTop 3 — Actual vs Predicted blended cost ($/1M tokens):")
print(top3_df[["model_name", TARGET, "predicted_cost"]].to_string(index=False))


Top 3 — Actual vs Predicted blended cost ($/1M tokens):
                                  model_name  blended_cost_usd_per_1m  predicted_cost
                      Gemini 3.1 Pro Preview                      4.5        6.824095
                 Gemini 3 Pro Preview (high)                      4.5        3.822762
Claude Opus 4.6 (Non-reasoning, High Effort)                     10.0        4.035530
